# Per-channel Grassmann distance entering the cross-branch attention

Five OTU test images are passed through **SAMUS** and **BUSSAM**. At each cross-branch site
(`image_encoder.blocks.2 / 5 / 8 / 11`) we capture the two feature maps that enter the
cross-branch module:

| Side | Tensor | Shape |
|---|---|---|
| ViT (query) | `blocks.<b>.norm1(x)` output | `(1, 32, 32, 768)` |
| CNN (key/value) | `cnnx` at the module call site | `(1, 32, 32, 768)` |

Each channel is a 1024-dim spatial vector (`32 x 32`). For every (model, image, block) we
build a **768 x 768 matrix** of pairwise channel distances. With
`cos(i, j) = <v_i, k_j> / (||v_i|| * ||k_j||)` the distance is selected by the
`GRASSMANN_METRIC` variable:

| Option | Name | Definition | Range |
|---|---|---|---|
| `1` | geodesic (arc length) | `arccos(|cos|)` | `[0, pi/2]` rad |
| `2` | projection (chordal) metric | `sqrt(1 - cos^2)` | `[0, 1]` |

Option 2 is the standard **projection metric** `sqrt(sum_i sin^2 theta_i)` on the
Grassmannian (Ham & Lee, 2008). The normalized subspace similarity used in the LoRA paper
(`1 - d^2 / p`) is deliberately **not** used.

**Layout:** 4 columns x 10 rows = **40 heatmaps**.

- Columns = blocks `2, 5, 8, 11`.
- Rows = 5 images x 2 models, interleaved: SAMUS img1, BUSSAM img1, SAMUS img2, ...
- Inside each heatmap: x = CNN channel `j` (0-767), y = ViT channel `i` (0-767).

In [ ]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch

warnings.filterwarnings('ignore', category=FutureWarning)
sys.path.insert(0, '/mnt/nvme0/home/utbt/KhoaVM/y_DG')

from DG.samus.model import build_samus
from DG.bussam.model import build_bussam
from utils.checkpoint import load_trainable
from utils.env import get_device
from utils.data import get_datasets

MODEL_TYPE = 'vit_b'
SOURCE = 'OTU'
BLOCKS = [2, 5, 8, 11]
MODELS_ORDER = ['SAMUS', 'BUSSAM']
N_IMAGES = 5
IMAGE_SIZE = 256

# Per-channel distance used in every heatmap below:
#   1 = geodesic / arc-length distance   arccos(|cos|)      in [0, pi/2] rad
#   2 = projection (chordal) metric      sqrt(1 - cos^2)    in [0, 1]
# Option 2 is the standard metric Grassmann distance, NOT the normalized
# subspace similarity (1 - d^2/p) from the LoRA paper (arXiv:2106.09685).
GRASSMANN_METRIC = 2

SAM_CKPT = 'weights/sam/sam_vit_b_01ec64.pth'
CKPTS = {
    'SAMUS':  f'weights/samus/{MODEL_TYPE}_samus_s_{SOURCE}_best.pth',
    'BUSSAM': f'weights/bussam/{MODEL_TYPE}_bussam_s_{SOURCE}_best.pth',
}
CB_MODULE = {'SAMUS': 'Space_Adapter', 'BUSSAM': 'MLP_Adapter1'}

DEVICE = get_device()
print('Device:', DEVICE)

_, _, test_ds = get_datasets(SOURCE, IMAGE_SIZE, transform=None)
SAMPLES = [test_ds[i][0] for i in range(N_IMAGES)]
print('Images:', N_IMAGES, '| tensor shape:', tuple(SAMPLES[0].shape))
print('Grassmann metric:', GRASSMANN_METRIC)

In [ ]:
MODELS = {}
for name in MODELS_ORDER:
    builder = build_samus if name == 'SAMUS' else build_bussam
    model = builder(checkpoint=SAM_CKPT, model_type=MODEL_TYPE,
                    image_size=IMAGE_SIZE)
    load_trainable(model, CKPTS[name], DEVICE)
    model.to(DEVICE).eval()
    MODELS[name] = model
    print(f'{name}: loaded {CKPTS[name]}')

In [ ]:
def branch_shapes(model, image, model_name):
    """Shapes of the two branch features entering the cross-branch module."""
    encoder = model.image_encoder
    captured = {b: {} for b in BLOCKS}
    hooks = []
    for b in BLOCKS:
        block = encoder.blocks[b]
        hooks.append(block.register_forward_pre_hook(
            lambda mo, inp, b=b: captured[b].__setitem__('cnnx', tuple(inp[1].shape))))
        hooks.append(block.norm1.register_forward_hook(
            lambda mo, inp, out, b=b: captured[b].__setitem__('vit', tuple(out.shape))))
        hooks.append(getattr(block, CB_MODULE[model_name]).register_forward_pre_hook(
            lambda mo, inp, b=b: captured[b].__setitem__(
                'cb', tuple(tuple(t.shape) for t in inp))))
    with torch.no_grad():
        encoder(image.unsqueeze(0).to(DEVICE))
    for h in hooks:
        h.remove()
    return captured

for name in MODELS_ORDER:
    captured = branch_shapes(MODELS[name], SAMPLES[0], name)
    print(f'{name} (image 1):')
    for b in BLOCKS:
        cb = captured[b].get('cb', '(module gated off; entering tensors shown)')
        print(f'  block {b}:  ViT norm1(x) {captured[b]["vit"]}  |  '
              f'CNN cnnx {captured[b]["cnnx"]}  |  CB inputs {cb}')

In [ ]:
def capture_entering_features(model, image, model_name):
    """(vit_feature, cnn_feature) per block, both (1024, 768) per image.

    vit_feature = norm1(x) output; cnn_feature = cnnx (module call site).
    When the cross-branch module fires we assert its inputs equal these.
    """
    encoder = model.image_encoder
    captured = {b: {} for b in BLOCKS}
    hooks = []
    for b in BLOCKS:
        block = encoder.blocks[b]
        hooks.append(block.register_forward_pre_hook(
            lambda mo, inp, b=b: captured[b].__setitem__('cnnx', inp[1].detach())))
        hooks.append(block.norm1.register_forward_hook(
            lambda mo, inp, out, b=b: captured[b].__setitem__('norm1', out.detach())))
        hooks.append(getattr(block, CB_MODULE[model_name]).register_forward_pre_hook(
            lambda mo, inp, b=b: captured[b].__setitem__(
                'cb_inputs', tuple(t.detach() for t in inp))))
    with torch.no_grad():
        encoder(image.unsqueeze(0).to(DEVICE))
    for h in hooks:
        h.remove()

    features = {}
    for b in BLOCKS:
        vit = captured[b]['norm1']
        cnn = captured[b]['cnnx']
        if 'cb_inputs' in captured[b]:
            q, k = captured[b]['cb_inputs'][0], captured[b]['cb_inputs'][1]
            assert torch.allclose(q, vit), f'block {b}: query != norm1(x)'
            assert torch.allclose(k, cnn), f'block {b}: key != cnnx'
        features[b] = (vit.reshape(-1, vit.shape[-1]).cpu(),
                       cnn.reshape(-1, cnn.shape[-1]).cpu())
    return features


def grassmann_metric_meta():
    """Colour limits / labels matching the selected `GRASSMANN_METRIC`."""
    if GRASSMANN_METRIC == 1:
        return dict(vmax=float(np.pi / 2),
                    ticks=[0.0, np.pi / 4, np.pi / 2],
                    ticklabels=['0', 'pi/4', 'pi/2'],
                    label='Grassmann distance (rad)',
                    tag='geodesic')
    if GRASSMANN_METRIC == 2:
        return dict(vmax=1.0,
                    ticks=[0.0, 0.5, 1.0],
                    ticklabels=['0', '0.5', '1'],
                    label='Grassmann distance (projection metric)',
                    tag='projection')
    raise ValueError(f'GRASSMANN_METRIC must be 1 or 2, got {GRASSMANN_METRIC!r}')


@torch.no_grad()
def per_channel_grassmann(vit, cnn, eps=1e-12):
    """768x768 Grassmann distance between ViT channels and CNN channels.

    vit, cnn: (1024, 768) feature matrices. Each channel is a 1024-dim vector,
    i.e. a 1-D subspace. With cos_ij = <v_i, k_j> / (||v_i|| ||k_j||):

      GRASSMANN_METRIC = 1 -> arccos(|cos_ij|)     geodesic / arc length (rad)
      GRASSMANN_METRIC = 2 -> sqrt(1 - cos_ij^2)   projection (chordal) metric
    """
    vit = vit.float().to(DEVICE)
    cnn = cnn.float().to(DEVICE)
    v = vit / vit.norm(dim=0, keepdim=True).clamp_min(eps)     # (1024, 768)
    k = cnn / cnn.norm(dim=0, keepdim=True).clamp_min(eps)     # (1024, 768)
    cos = (v.T @ k).clamp(-1.0, 1.0).abs()                     # (768, 768)
    if GRASSMANN_METRIC == 1:
        d = torch.arccos(cos)
    elif GRASSMANN_METRIC == 2:
        d = torch.sqrt((1.0 - cos ** 2).clamp_min(0.0))
    else:
        raise ValueError(f'GRASSMANN_METRIC must be 1 or 2, got {GRASSMANN_METRIC!r}')
    return d.cpu().numpy()


DIST = {}   # (model, image_idx, block) -> (768, 768) array
for i, image in enumerate(SAMPLES):
    for name in MODELS_ORDER:
        features = capture_entering_features(MODELS[name], image, name)
        for b in BLOCKS:
            vit_feat, cnn_feat = features[b]
            DIST[(name, i, b)] = per_channel_grassmann(vit_feat, cnn_feat)
    print(f'  processed image {i + 1}/{N_IMAGES}')

META = grassmann_metric_meta()
sample = DIST[('SAMUS', 0, BLOCKS[0])]
print(f'per-channel matrix shape: {sample.shape} | metric: {META["tag"]} | '
      f'min/mean/max = {sample.min():.3f}/{sample.mean():.3f}/{sample.max():.3f}')

In [ ]:
import matplotlib.gridspec as gridspec

META = grassmann_metric_meta()
n_rows = N_IMAGES * 2
n_cols = len(BLOCKS)

# Dedicated GridSpec column for the colorbar -> it lives in its own axes,
# completely outside every heatmap axes (nothing overlaps).
fig = plt.figure(figsize=(2.8 * n_cols + 0.9, 2.6 * n_rows),
                 layout='constrained')
gs = fig.add_gridspec(n_rows, n_cols + 1,
                      width_ratios=[1.0] * n_cols + [0.045])

axes = np.empty((n_rows, n_cols), dtype=object)
for r in range(n_rows):
    for c in range(n_cols):
        axes[r, c] = fig.add_subplot(gs[r, c])
cax = fig.add_subplot(gs[:, -1])          # colorbar column, outside the grid

cmap = plt.get_cmap('magma')
im = None
for i in range(N_IMAGES):
    for j, name in enumerate(MODELS_ORDER):
        row = 2 * i + j
        for c, b in enumerate(BLOCKS):
            ax = axes[row, c]
            im = ax.imshow(DIST[(name, i, b)], cmap=cmap, vmin=0.0,
                           vmax=META['vmax'], origin='upper', aspect='auto',
                           interpolation='nearest')
            ax.set_xticks([0, 767])
            ax.set_yticks([0, 767])
            ax.tick_params(labelsize=7)
            ax.set_title(f'{name} | image {i + 1} | block {b}',
                         fontsize=10, fontweight='bold')
            if c == 0:
                ax.set_ylabel(f'{name}\nimage {i + 1}\n\nViT channel i',
                              fontsize=9, fontweight='bold')
            else:
                ax.set_ylabel('')
            if row == n_rows - 1:
                ax.set_xlabel('CNN channel j', fontsize=8)

cbar = fig.colorbar(im, cax=cax)
cbar.set_ticks(META['ticks'])
cbar.set_ticklabels(META['ticklabels'])
cbar.set_label(META['label'], fontsize=11, fontweight='bold')

fig.suptitle(
    'Per-channel Grassmann distance entering the cross-branch attention\n'
    f'metric {GRASSMANN_METRIC} ({META["tag"]}) | '
    f'{MODEL_TYPE} | source {SOURCE} | {N_IMAGES} test images | blocks {BLOCKS} | '
    f'{n_rows} rows x {n_cols} cols of 768x768 maps',
    fontsize=14, fontweight='bold')  # y handled by constrained layout

os.makedirs('saved_plots', exist_ok=True)
out = (f'saved_plots/Grassmann_perChannel_768x768_{META["tag"]}_'
       f'{MODEL_TYPE}_{SOURCE}.png')
plt.savefig(out, dpi=110, bbox_inches='tight', facecolor='white')
print('Saved:', out)
plt.show()

## Same per-channel heatmaps after truncating branch(es) (lowest sigma only)

New cell — the cells above are unchanged, no weight or sample is modified. The selected
`GRASSMANN_METRIC` (1 = geodesic, 2 = projection metric) is applied unchanged. For each
(model, image, block) the selected branch feature `(1024, 768)` is SVD-truncated by zeroing
its **100 smallest singular values** together with their `U` columns and `V^T` rows, then
reconstructed:

- `TRUNC_BRANCH = 'cnn'`  → truncate the CNN branch only
- `TRUNC_BRANCH = 'vit'`  → truncate the ViT branch only
- `TRUNC_BRANCH = 'both'` → truncate **both** branches

One figure is produced per mode.

In [ ]:
def truncate_lowest_sigma(X, n_remove=100):
    """Zero the n_remove smallest singular values of X (1024, 768) and the
    matching U columns / V^T rows, then reconstruct.

    X = U diag(S) V^T  ->  zero S[k:], U[:, k:], V^T[k:, :],  k = 768 - n_remove
    """
    Xf = X.to(DEVICE).float()
    U, S, Vh = torch.linalg.svd(Xf, full_matrices=False)
    k = max(S.numel() - n_remove, 0)
    S_z = S.clone();   S_z[k:] = 0.0
    U_z = U.clone();   U_z[:, k:] = 0.0
    Vh_z = Vh.clone(); Vh_z[k:, :] = 0.0
    X_trunc = (U_z * S_z) @ Vh_z
    return X_trunc, S.detach().cpu(), k


def plot_grid(DIST, title, out, show=False):
    """10 x 4 grid of 768 x 768 per-channel Grassmann heatmaps."""
    import matplotlib.gridspec as gridspec
    META = grassmann_metric_meta()
    n_rows, n_cols = N_IMAGES * 2, len(BLOCKS)
    fig = plt.figure(figsize=(2.8 * n_cols + 0.9, 2.6 * n_rows),
                     layout='constrained')
    gs = fig.add_gridspec(n_rows, n_cols + 1,
                          width_ratios=[1.0] * n_cols + [0.045])
    axes = np.empty((n_rows, n_cols), dtype=object)
    for r in range(n_rows):
        for c in range(n_cols):
            axes[r, c] = fig.add_subplot(gs[r, c])
    cax = fig.add_subplot(gs[:, -1])
    cmap = plt.get_cmap('magma')
    im = None
    for i in range(N_IMAGES):
        for j, name in enumerate(MODELS_ORDER):
            row = 2 * i + j
            for c, b in enumerate(BLOCKS):
                ax = axes[row, c]
                im = ax.imshow(DIST[(name, i, b)], cmap=cmap, vmin=0.0,
                               vmax=META['vmax'], origin='upper', aspect='auto',
                               interpolation='nearest')
                ax.set_xticks([0, 767]); ax.set_yticks([0, 767])
                ax.tick_params(labelsize=7)
                ax.set_title(f'{name} | image {i + 1} | block {b}',
                             fontsize=10, fontweight='bold')
                if c == 0:
                    ax.set_ylabel(f'{name}\nimage {i + 1}\n\nViT channel i',
                                  fontsize=9, fontweight='bold')
                else:
                    ax.set_ylabel('')
                if row == n_rows - 1:
                    ax.set_xlabel('CNN channel j', fontsize=8)
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_ticks(META['ticks'])
    cbar.set_ticklabels(META['ticklabels'])
    cbar.set_label(META['label'], fontsize=11, fontweight='bold')
    fig.suptitle(title, fontsize=14, fontweight='bold')
    os.makedirs('saved_plots', exist_ok=True)
    plt.savefig(out, dpi=110, bbox_inches='tight', facecolor='white')
    print('Saved:', out)
    if show:
        plt.show()          # show only the last mode inline
    plt.close(fig)          # free the figure so figures never pile up


N_REMOVE = 758                          # keep S[0 : 768 - N_REMOVE] (here 3 sigma)
MODES = ('both',)                       # options: 'cnn', 'vit', 'both'
plt.close('all')                        # drop any figures left open by earlier runs
META = grassmann_metric_meta()

for mi, TRUNC_BRANCH in enumerate(MODES):
    DIST_TRUNC = {}
    info = None
    for i, image in enumerate(SAMPLES):
        for name in MODELS_ORDER:
            features = capture_entering_features(MODELS[name], image, name)
            for b in BLOCKS:
                vit_feat, cnn_feat = features[b]        # both (1024, 768), unchanged
                vit_use, cnn_use = vit_feat, cnn_feat
                S_cnn = S_vit = k = None
                if TRUNC_BRANCH in ('cnn', 'both'):
                    cnn_use, S_cnn, k = truncate_lowest_sigma(cnn_feat, N_REMOVE)
                if TRUNC_BRANCH in ('vit', 'both'):
                    vit_use, S_vit, k = truncate_lowest_sigma(vit_feat, N_REMOVE)
                DIST_TRUNC[(name, i, b)] = per_channel_grassmann(vit_use, cnn_use)
                if info is None and name == 'SAMUS' and i == 0 and b == BLOCKS[0]:
                    info = {}
                    if S_cnn is not None:
                        info['cnn'] = (S_cnn, k)
                    if S_vit is not None:
                        info['vit'] = (S_vit, k)
    print(f"--- truncate_branch='{TRUNC_BRANCH}': zeroed the {N_REMOVE} lowest "
          f"sigma + matching U/V")
    for br, (S, k) in info.items():
        print(f"    {br.upper():>3}: sigma_max={float(S.max()):.4f} | removed sigma "
              f"mean={float(S[k:].mean()):.6f} (max {float(S[k:].max()):.6f}) | "
              f"kept S[0:{k}] of {S.numel()}")
    d = DIST_TRUNC[('SAMUS', 0, BLOCKS[0])]
    print(f"    distance (SAMUS, img 1, block {BLOCKS[0]}, "
          f"metric {GRASSMANN_METRIC}/{META['tag']}): "
          f"min/mean/max = {d.min():.3f}/{d.mean():.3f}/{d.max():.3f}")
    plot_grid(
        DIST_TRUNC,
        f'Per-channel Grassmann distance ({META["tag"]}), lowest {N_REMOVE} sigma of '
        f'{TRUNC_BRANCH.upper()} branch(es) zeroed\n'
        f'metric {GRASSMANN_METRIC} | {MODEL_TYPE} | source {SOURCE} | '
        f'{N_IMAGES} test images | blocks {BLOCKS}',
        f'saved_plots/Grassmann_perChannel_768x768_truncLowest{N_REMOVE}_'
        f'{META["tag"]}_{TRUNC_BRANCH}_{MODEL_TYPE}_{SOURCE}.png',
        show=(mi == len(MODES) - 1))